### 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

**This lab attaches to a shared APIM instance** (see `main.bicep`), so cleanup is different from a lab that deploys its own APIM: deleting this lab's own resource group is NOT enough, because the real resources — this lab's 2 APIs, 2 backends, 1 Product and its subscriptions — live inside the SHARED APIM's resource group, alongside other labs' resources. The cell below deletes ONLY the `gemini-models-`-prefixed resources it created; it never touches the shared APIM service itself, the shared Log Analytics workspace / App Insights, or any other lab's resources.

In [ ]:
import os, sys
sys.path.insert(1, '../../shared')  # add the shared directory to the Python path
import utils

deployment_name = os.path.basename(os.path.dirname(globals()['__vsc_ipynb_file__']))
resource_group = f"lab-{deployment_name}"

shared_apim_name = "apim-shared-pdcibwky2f5ms"
shared_apim_resource_group_name = "rg-shared-apim-gateway-V2"

# Must match the subscriptions config used at deploy time in gemini-models.ipynb —
# update this if you changed it there, so every subscription actually gets deleted.
apim_subscriptions_config = [{"name": "subscription1"}, {"name": "subscription2"}, {"name": "subscription3"}]

In [ ]:
# 1. Delete the Product (this also removes its product<->API links, and — with
#    --delete-subscriptions true — the subscriptions scoped to it).
utils.run(
    f"az apim product delete --resource-group {shared_apim_resource_group_name} "
    f"--service-name {shared_apim_name} --product-id gemini-models-product "
    f"--delete-subscriptions true --yes",
    "Deleted gemini-models-product (and its subscriptions)",
    "Failed to delete gemini-models-product — check it isn't already gone, or delete it manually in the portal"
)

# 2. Delete the 2 APIs this lab created.
for api_id in ["gemini-models-inference-api", "gemini-models-openai-api"]:
    utils.run(
        f"az apim api delete --resource-group {shared_apim_resource_group_name} "
        f"--service-name {shared_apim_name} --api-id {api_id} --yes",
        f"Deleted {api_id}",
        f"Failed to delete {api_id} — check it isn't already gone, or delete it manually in the portal"
    )

# 3. Delete the 2 backends this lab created.
for backend_id in ["gemini-models-backend", "gemini-models-backend-openai"]:
    utils.run(
        f"az apim backend delete --resource-group {shared_apim_resource_group_name} "
        f"--service-name {shared_apim_name} --backend-id {backend_id} --yes",
        f"Deleted {backend_id}",
        f"Failed to delete {backend_id} — check it isn't already gone, or delete it manually in the portal"
    )

# Belt-and-braces: in case any subscription survived (e.g. --delete-subscriptions
# didn't catch one), delete them individually too — safe to run even if already gone.
for subscription in apim_subscriptions_config:
    sub_id = f"gemini-models-{subscription['name']}"
    utils.run(
        f"az apim subscription delete --resource-group {shared_apim_resource_group_name} "
        f"--service-name {shared_apim_name} --sid {sub_id} --yes",
        f"Deleted subscription {sub_id}",
        f"{sub_id} already gone (expected if step 1 already removed it)"
    )

# 5. Delete the per-subscription Google API key Named Values (and the
#    default/fallback one) created for quota isolation — see
#    gemini-shared-resources.bicep. Safe to run even if some are already gone.
named_value_ids = [f"gemini-models-key-{s['name']}" for s in apim_subscriptions_config] + ["gemini-models-key-default"]
for named_value_id in named_value_ids:
    utils.run(
        f"az apim nv delete --resource-group {shared_apim_resource_group_name} "
        f"--service-name {shared_apim_name} --named-value-id {named_value_id} --yes",
        f"Deleted named value {named_value_id}",
        f"{named_value_id} already gone (or never created)"
    )

In [ ]:
# 4. Finally, remove this lab's own (empty) resource group — it never held any
#    real resources, it only existed as the deployment's nominal target.
utils.cleanup_resources(deployment_name, resource_group_name=resource_group)